# SAMA-NMC — idea-stage proof (SIH26099)

**AI proposes, rules dispose, humans approve, the ledger remembers.**

This notebook runs the real SAMA-NMC core engine end to end on a seeded synthetic benchmark
(5 CPSE house styles, noise, ~15% trap twins) and on a held-out *unseen-noise* set. It shows:

1. a trap twin (Class 150 vs Class 300) **rejected**;
2. *carbon steel* vs *A216 WCB* sent to an **engineer** (vaguer, not different);
3. a NACE vs non-NACE pair sent to **review**;
4. a weld-neck flange pair **auto-merged** with its Evidence Certificate and a National Material Code;
5. STD vs SCH40 — the same wall at 2 inch, a conflict at 12 inch;
6. an ordinary fuzzy matcher vs SAMA-NMC on **the same candidate pairs** — wrong-merge counts measured, not promised.

**Honesty notes.** All data here is synthetic (labelled). Engineering tables are `unverified` drafts pending
domain-reviewer sign-off. Precision figures are benchmark measurements, never production guarantees.

In [1]:
import sys, json, math
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "backend"))
import pandas as pd
from IPython.display import display, HTML
from sama.pipeline import run
from sama.certificate import build_certificate
from sama.export_demo import SHOWCASE, _find
from sama.nmc import validate
pd.set_option("display.max_colwidth", 80)

main = run(ROOT / "data/synthetic/demo", evidence_scope="synthetic")
unseen = run(ROOT / "data/synthetic/unseen", evidence_scope="unseen_noise", run_id="RUN-0002",
             gate=main.gate, thr_E=main.thr_E, t_base=main.t_base)
print(f"records: {len(main.raws):,}   candidate pairs: {len(main.pairs):,}   timings: {main.timings}")
print(f"unseen-noise records: {len(unseen.raws):,}   candidate pairs: {len(unseen.pairs):,}")

records: 10,137   candidate pairs: 362,407   timings: {'standardise_s': 3.43, 'block_compare_s': 146.96, 'score_zone_s': 5.96, 'total_s': 157.06, 'n_records': 10137, 'n_pairs': 362407}
unseen-noise records: 4,022   candidate pairs: 63,373


## 1. The worked examples (PRD §3.5) — every decision with its reason

In [2]:
rows = []
for _id, title, lt, rt in SHOWCASE:
    a, b = _find(main, lt), _find(main, rt)
    p = main.score_pair(a, b)
    crit = "; ".join(f"{r.property}: {r.state}" for r in p.comparison if r.critical and r.state != "agree") or "all critical agree"
    rows.append({"case": title, "left": lt, "right": rt, "zone": p.zone, "step": p.zone_step,
                 "why": p.zone_reason, "non-agreeing critical attributes": crit})
display(pd.DataFrame(rows))

,case,left,right,zone,step,why,non-agreeing critical attributes
0,"Carbon steel vs A216 WCB — vaguer, not different",VALVE GATE 2IN 150# CS FLGD,GATE VLV 50MM CL150 A216 WCB FLANGED END,REVIEW,6,engineered item (Tier R) → engineer review,body_material: less_specific
1,End connection stated on one side only,VALVE GATE 2IN 150# CS FLGD,"Valve, Gate, NPS 2, Class 150, Carbon Steel, RF",REVIEW,6,engineered item (Tier R) → engineer review,body_material: less_specific; end_connection: right_missing
2,Trap twin — Class 150 vs Class 300,GATE VLV 50MM CL150 A216 WCB FLGD RF,GATE VLV 50MM CL300 A216 WCB FLGD RF,REJECT,2,critical conflict: pressure_class,pressure_class: conflict
3,"Weld-neck flange — standard part, auto-merged with proof","FLANGE WN 2"" CL150 RF SCH40 A105","WELD NECK FLG, NPS 2, 150#, RF, S40, ASTM A105",AUTO_MERGE,8,every critical attribute agrees and gate score clears the measured threshold...,all critical agree
4,STD vs SCH40 at 2 inch — same wall via the size table,PIPE SMLS NPS 2 STD A106 GR B,"PIPE SEAMLESS 2"" SCH40 ASTM A106 B",REVIEW,7,agreement via unreviewed size table: wall,all critical agree
5,"Same attributes, but one record says NACE",BALL VLV 1IN CL300 SS316 NACE,"BALL VALVE 1"" 300# SS316",REVIEW,3,risk word: NACE,end_connection: both_missing


## 2. Size-dependent designations: STD vs SCH40

ASME schedule designations mean different wall thicknesses at different sizes. SAMA-NMC compares the
*table-resolved wall thickness*, not the label. While the size table is `unverified`, an agreement that
depends on it is capped at review (R-09).

In [3]:
from sama.decide import build_pair, zone
from sama.pipeline import standardise
from sama.contracts import RawRecord
def adhoc(lt, rt):
    raws = [RawRecord(record_id="X:1", cpse="X", matnr="1", maktx=lt), RawRecord(record_id="Y:2", cpse="Y", matnr="2", maktx=rt)]
    _, recs = standardise(raws, main.config)
    p = build_pair(recs["X:1"], recs["Y:2"], main.config); p.gate_score = 1.0
    z, step, why = zone(p, main.thr_E, main.reject_thr, main.config)
    wall = next(r for r in p.comparison if r.property == "wall")
    return {"left": lt, "right": rt, "wall": f"{wall.left} vs {wall.right}: {wall.state}" + (" (via size table)" if wall.via else ""),
            "zone": z, "why": why}
display(pd.DataFrame([adhoc("PIPE SMLS NPS 2 STD A106 GR B", 'PIPE SEAMLESS 2" SCH40 ASTM A106 B'),
                      adhoc("PIPE SMLS NPS 12 STD A106 GR B", 'PIPE SEAMLESS 12" SCH40 ASTM A106 B')]))

,left,right,wall,zone,why
0,PIPE SMLS NPS 2 STD A106 GR B,"PIPE SEAMLESS 2"" SCH40 ASTM A106 B",STD vs SCH40: agree (via size table),REVIEW,agreement via unreviewed size table: wall
1,PIPE SMLS NPS 12 STD A106 GR B,"PIPE SEAMLESS 12"" SCH40 ASTM A106 B",STD vs SCH40: conflict (via size table),REJECT,critical conflict: wall


## 3. Evidence Certificate for the auto-merged weld-neck flange pair

In [4]:
a, b = _find(main, 'FLANGE WN 2" CL150 RF SCH40 A105'), _find(main, "WELD NECK FLG, NPS 2, 150#, RF, S40, ASTM A105")
key = tuple(sorted((a, b)))
main.score_pair(a, b)
cert = build_certificate(main, key, "CERT-000004", scopes={"synthetic": main.metrics, "unseen_noise": unseen.metrics})
c = cert.model_dump()
print("NMC:", c["subject"]["nmc"], "| check digit valid:", validate(c["subject"]["nmc"]) if c["subject"]["nmc"] else None)
display(pd.DataFrame([r for r in c["comparison"]])[["property", "critical", "left", "right", "state"]])
print(json.dumps({k: c[k] for k in ("decision", "tier", "threshold", "measured_precision", "config_status", "limitations")},
                 indent=1, default=str))

NMC: NMC:1201-0000589-3 | check digit valid: True


,property,critical,left,right,state
0,size_nps,True,2,2,agree
1,pressure_class,True,CL150,CL150,agree
2,face,True,RF,RF,agree
3,bore_wall,True,SCH40,SCH40,agree
4,material,True,ASTM A105,ASTM A105,agree
5,standard,False,None,None,both_missing


{
 "decision": {
  "zone": "AUTO_MERGE",
  "zone_step": 8,
  "reason": "every critical attribute agrees and gate score clears the measured threshold \u2192 auto-merge",
  "status": "approved",
  "approvals": [
   {
    "user": "system",
    "role": "auto"
   }
  ]
 },
 "tier": {
  "value": "E",
  "reasons": [
   "class 1201 default E"
  ]
 },
 "threshold": {
  "applies_to": "gate_score",
  "value": 0.752981,
  "is_demo_value": false,
  "min_n": 598,
  "evidence_scope": "synthetic"
 },
 "measured_precision": [
  {
   "evidence_scope": "synthetic",
   "n": 619,
   "false_merges": 0,
   "upper_bound_95": 0.00483
  },
  {
   "evidence_scope": "unseen_noise",
   "n": 188,
   "false_merges": 0,
   "upper_bound_95": 0.01581
  }
 ],
 "config_status": {
  "abbreviations": "unverified",
  "hierarchies": "unverified",
  "residual_vocab": "draft",
  "risk_words": "draft",
  "schedules": "unverified",
  "tiers": "draft",
  "units": "unverified"
 },
 "limitations": [
  "Precision figures are benchma

## 4. Ordinary fuzzy matching vs SAMA-NMC — same data, same candidate pairs

The baseline is what a typical fuzzy matcher does: `mean(char TF-IDF cosine, token_set_ratio/100)`, auto-merge
above a threshold. Its threshold is the one that **maximises its own F1 on the calibration split**, so it is shown
at its best. Counts below are on the held-out **test split**. This compares *decision safety on identical
candidate pairs*; it is not an end-to-end retrieval benchmark.

In [5]:
def table(m):
    return pd.DataFrame({mode: {"wrong auto-merges": d["auto_wrong"], "correct auto-merges": d["auto_correct"],
                                "sent to review": d["review"], "rejected": d["reject"],
                                "true matches rejected": d["reject_true_match"],
                                "auto-merge precision": d["auto_precision"], "95% upper bound on error": d["upper_bound_95"],
                                "total recall (auto+review)": d["total_recall"]}
                         for mode, d in m["modes"].items()})
print("SYNTHETIC test split —", main.metrics["n_pairs"], "candidate pairs; baseline t =", main.metrics["threshold"]["t_base"],
      "(F1", main.metrics["threshold"]["t_base_f1"], ")")
display(table(main.metrics))
print("UNSEEN-NOISE set —", unseen.metrics["n_pairs"], "candidate pairs (held-out style, abbreviations, filler)")
display(table(unseen.metrics))

SYNTHETIC test split — 17243 candidate pairs; baseline t = 0.9287 (F1 0.4022 )


,sama,baseline,model_only
wrong auto-merges,0.00000,848.00000,79.00000
correct auto-merges,619.00000,1027.00000,1657.00000
sent to review,2658.00000,15368.00000,1325.00000
rejected,13966.00000,0.00000,14182.00000
true matches rejected,0.00000,0.00000,33.00000
auto-merge precision,1.00000,0.54770,0.95450
95% upper bound on error,0.00483,0.47149,0.05461
total recall (auto+review),1.00000,1.00000,0.98650


UNSEEN-NOISE set — 63373 candidate pairs (held-out style, abbreviations, filler)


,sama,baseline,model_only
wrong auto-merges,0.00000,778.00000,16535.0000
correct auto-merges,188.00000,270.00000,1548.0000
sent to review,24884.00000,62325.00000,6769.0000
rejected,38301.00000,0.00000,38521.0000
true matches rejected,65.00000,0.00000,86.0000
auto-merge precision,1.00000,0.25760,0.0856
95% upper bound on error,0.01581,0.76454,0.9178
total recall (auto+review),0.96930,1.00000,0.9594


In [6]:
print("Decisions by pair type (synthetic test split):")
display(pd.DataFrame(main.metrics["by_pair_type"]).fillna(0).astype(int).T)

Decisions by pair type (synthetic test split):


,sama:REVIEW,baseline:REVIEW,model_only:AUTO_MERGE,model_only:REVIEW,model_only:REJECT,sama:AUTO_MERGE,baseline:AUTO_MERGE,sama:REJECT
desig,9,9,9,0,0,0,0,0
downgrade,439,463,29,405,29,24,0,0
identical,1375,943,1619,347,4,595,1027,0
mpn_suffix,79,49,0,9,70,0,30,0
random,540,13162,20,420,13483,0,761,13383
riskword,80,78,58,22,0,0,2,0
trap,136,664,1,122,596,0,55,583


In [7]:
wrong = [k for k in main.pairs if main.pair_split(k) == "test" and main.zones["baseline"][k] == "AUTO_MERGE" and not main.is_match(k)]
ex = []
for k in sorted(wrong, key=lambda k: -main.pairs[k].baseline_score)[:8]:
    p = main.pairs[k]
    ex.append({"left": main.raws[k[0]].maktx, "right": main.raws[k[1]].maktx, "type": main.pair_types.get(k, "random"),
               "fuzzy score": round(p.baseline_score, 3), "fuzzy": "AUTO_MERGE", "SAMA-NMC": p.zone, "SAMA reason": p.zone_reason})
print("Examples the fuzzy matcher merged that are NOT the same item:")
display(pd.DataFrame(ex))

Examples the fuzzy matcher merged that are NOT the same item:


,left,right,type,fuzzy score,fuzzy,SAMA-NMC,SAMA reason
0,FLANGE SO 80MM 150# FF SS,"Flange, Slip On, NPS 3, Class 150, Stainless Steel, FF",random,1.0,AUTO_MERGE,REVIEW,less specific: material
1,"Flange, Weld Neck, NPS 2, Class 600, SCH 40S, Stainless Steel, FF",FLG WELD NECK STAINLESS STEEL DN50 FF SCH40S CL 600,random,1.0,AUTO_MERGE,REVIEW,engineered item (Tier R) → engineer review
2,VALVE BALL 10IN 600# SS316L SW,VALVE BALL 10IN 600# SS316L SW,mpn_suffix,1.0,AUTO_MERGE,REVIEW,engineered item (Tier R) → engineer review
3,BALL VLV 250MM CL600 SS316L SOCKETWELD END,BALL VLV 250MM CL600 SS316L SOCKETWELD END,mpn_suffix,1.0,AUTO_MERGE,REVIEW,engineered item (Tier R) → engineer review
4,GATE VLV 15MM CL300 SS304 BUTTWELD END,GATE VLV BUTTWELD END CL300 SS304 15MM,mpn_suffix,1.0,AUTO_MERGE,REVIEW,engineered item (Tier R) → engineer review
5,"Valve, Gate, NPS 1/2, Class 300, SS304, BW",VLV GATE CL 300 DN15 BUTTWELD SS304,random,1.0,AUTO_MERGE,REVIEW,engineered item (Tier R) → engineer review
6,VALVE CHECK 3IN 600# CS THRD,"CHECK VALVE 3"" CARBON STEEL 600 LB THRD",trap,1.0,AUTO_MERGE,REVIEW,engineered item (Tier R) → engineer review
7,"Valve, Globe, NPS 2, Class 600, ASTM A216 WCB, Buttweld","Valve, Globe, NPS 2, Class 600, ASTM A216 WCB, Buttweld",mpn_suffix,1.0,AUTO_MERGE,REVIEW,engineered item (Tier R) → engineer review


## 5. Hard safety gates (PRD §11) and threshold evidence

In [8]:
display(pd.DataFrame({"synthetic": main.metrics["safety"], "unseen_noise": unseen.metrics["safety"]}))
print(json.dumps(main.metrics["threshold"], indent=1))
print("blocking recall (synthetic test):", main.metrics["blocking"])

,synthetic,unseen_noise
auto_false_merges,0,0
auto_trap,0,0
auto_riskword,0,0
auto_mpn_suffix,0,0
auto_downgrade_without_mpn,0,0
passed,True,True


{
 "thr_E": 0.752981,
 "is_demo": false,
 "min_n": 598,
 "n_eligible": 936,
 "n_above": 936,
 "errors": 0,
 "upper_bound_95": 0.003195452308302631,
 "stopped_reason": "reached lowest candidate",
 "t_base": 0.9287,
 "t_base_f1": 0.4022
}
blocking recall (synthetic test): {'true_pairs': 2486, 'found': 2442, 'recall': 0.9823}


**Reading the threshold line.** The Tier E auto-merge threshold is chosen on the calibration split so that the
95% upper bound on the false-merge rate stays within 0.5%, which needs at least 598 error-free auto-eligible pairs.
If the calibration split is too small, Tier E does not auto-merge by score at all and the report says so.

## 6. National Material Codes minted

In [9]:
rows = []
for code, v in list(main.nmcs.items()):
    if v["kind"] == "cluster" and len({main.raws[r].cpse for r in v["members"]}) >= 2:
        rows.append({"NMC": code, "valid": validate(code), "short text": v["short_text"],
                     "legacy codes": ", ".join(f"{main.raws[r].cpse}:{main.raws[r].matnr}" for r in v["members"])})
print(f"NMCs minted: {len(main.nmcs):,} ({sum(v['kind']=='cluster' for v in main.nmcs.values())} verified clusters); "
      f"records pending engineer review (no code yet, by design): {len(main.pending):,}")
display(pd.DataFrame(rows[:10]))

NMCs minted: 1,171 (1165 verified clusters); records pending engineer review (no code yet, by design): 9,676


,NMC,valid,short text,legacy codes
0,NMC:1203-0000001-8,True,FLG BLIND 10 CL300 RF ASTM A105,"CPSE_A:10001317, CPSE_C:4500-941"
1,NMC:1102-0000002-X,True,VALVE GLOBE 6 CL600 SS316L THREADED,"CPSE_A:10003341, CPSE_B:M-40134, CPSE_C:4600-272, CPSE_D:000000000026870804"
2,NMC:1202-0000003-7,True,FLG SLIP ON 10 CL150 FF ASTM A350 LF2,"CPSE_A:10006626, CPSE_A:10892922, CPSE_B:M-96007, CPSE_D:000000000079240333"
3,NMC:1203-0000004-2,True,FLG BLIND 3/4 CL300 RTJ ASTM A105,"CPSE_A:10009343, CPSE_D:000000000029570670"
4,NMC:1101-0000005-7,True,VALVE GATE 3 CL600 SS316 THREADED,"CPSE_A:10009566, CPSE_C:4600-922"
5,NMC:1101-0000006-5,True,VALVE GATE 10 CL300 ASTM A350 LF2 BUTTW…,"CPSE_A:10010049, CPSE_A:10880395, CPSE_B:M-93696, CPSE_B:M-98961, CPSE_D:000..."
6,NMC:1102-0000007-0,True,VALVE GLOBE 10 CL600 ASTM A350 LF2 SOCK…,"CPSE_A:10010286, CPSE_B:M-32088, CPSE_E:MAT/755405"
7,NMC:1201-0000008-0,True,FLG WELD NECK 12 CL300 FF SCH80 ASTM A3…,"CPSE_A:10010417, CPSE_B:M-99903, CPSE_C:4520-368, CPSE_E:MAT/248207"
8,NMC:1201-0000009-9,True,FLG WELD NECK 2 CL150 FF 40S SS316,"CPSE_A:10010519, CPSE_C:4510-681"
9,NMC:1202-0000010-9,True,FLG SLIP ON 1 CL150 RF ASTM A105,"CPSE_A:10010783, CPSE_B:M-43050"
